# Phase II_03: Metadata Extraction

**Objective**: Extract and map CaBuAr tile metadata

**Input**: CaBuAr HDF5 from Google Drive

**Output**: JSON database with tile metadata

## Setup: Mount Google Drive

In [ ]:
%pip install -q h5py
import sys
import json
import numpy as np
from collections import defaultdict
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '/content/RETINNA/src')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

# Find CaBuAr HDF5 in Drive
# Path: /content/drive/MyDrive/RETINNA_DATA/cabuaur/512x512.hdf5
hdf5_path = Path('/content/drive/MyDrive/RETINNA_DATA/cabuaur/512x512.hdf5')

if hdf5_path.exists():
    print(f"✓ Found CaBuAr: {hdf5_path}")
    file_size_gb = hdf5_path.stat().st_size / (1024**3)
    print(f"  Size: {file_size_gb:.1f} GB")
else:
    raise FileNotFoundError(f"CaBuAr HDF5 not found at {hdf5_path}")

print(f"\n✓ Ready to extract metadata")

## Extract metadata from HDF5

In [ ]:
import h5py

def extract_cabuaur_metadata(hdf5_path):
    """
    Extract available metadata from CaBuAr HDF5 file.
    """
    metadata = {}
    fire_map = defaultdict(list)
    
    with h5py.File(str(hdf5_path), 'r') as f:
        all_keys = list(f.keys())
        print(f"Total samples in HDF5: {len(all_keys)}")
        
        for idx, key in enumerate(all_keys):
            if (idx + 1) % 100 == 0:
                print(f"  Processed {idx + 1}/{len(all_keys)}...")
            
            sample = f[key]
            
            # Extract available metadata
            uuid = key.split('_')[0]
            tile_idx = int(key.split('_')[-1])
            fold = int(sample.attrs.get('fold', -1))
            comments = list(sample.attrs.get('comments', []))
            
            metadata[key] = {
                'uuid': uuid,
                'tile_idx': tile_idx,
                'fold': fold,
                'comments': comments,
                'fold_name': ['val', 'train', 'train', 'train', 'train'][fold] if fold >= 0 else 'unknown'
            }
            
            fire_map[uuid].append(key)
    
    return metadata, dict(fire_map)

# Extract metadata
print(f"Extracting metadata from HDF5...")
metadata, fire_map = extract_cabuaur_metadata(hdf5_path)

print(f"\n✓ Extracted {len(metadata)} samples")
print(f"✓ Found {len(fire_map)} unique fires")
print(f"\nFire statistics:")
tile_counts = [len(tiles) for tiles in fire_map.values()]
print(f"  Avg tiles per fire: {np.mean(tile_counts):.1f}")
print(f"  Min tiles: {np.min(tile_counts)}, Max tiles: {np.max(tile_counts)}")

## Build JSON database

In [ ]:
def build_metadata_database(metadata, fire_map):
    """
    Build structured JSON database of all samples.
    """
    database = {
        'metadata': {
            'created': datetime.now().isoformat(),
            'total_fires': len(fire_map),
            'total_samples': len(metadata),
            'note': 'UUIDs are fire identifiers; coordinate/date info to be added from external sources'
        },
        'fires': []
    }
    
    for uuid, sample_keys in fire_map.items():
        fire_entry = {
            'uuid': uuid,
            'fire_name': None,
            'fire_id': None,
            'pre_date': None,
            'post_date': None,
            'location': {
                'bbox_north': None,
                'bbox_south': None,
                'bbox_east': None,
                'bbox_west': None,
                'center_lat': None,
                'center_lon': None,
                'utm_zone': None
            },
            'num_tiles': len(sample_keys),
            'split': {'train': 0, 'val': 0},
            'samples': []
        }
        
        # Count splits
        for key in sample_keys:
            fold_name = metadata[key]['fold_name']
            if fold_name in fire_entry['split']:
                fire_entry['split'][fold_name] += 1
        
        # Add sample details
        for key in sorted(sample_keys):
            fire_entry['samples'].append({
                'sample_id': key,
                'tile_idx': metadata[key]['tile_idx'],
                'fold': metadata[key]['fold_name'],
                'comments': metadata[key]['comments']
            })
        
        database['fires'].append(fire_entry)
    
    return database

print("Building metadata database...")
database = build_metadata_database(metadata, fire_map)

print(f"\n✓ Database created:")
print(f"  Total fires: {database['metadata']['total_fires']}")
print(f"  Total samples: {database['metadata']['total_samples']}")

## Save to Drive

In [ ]:
# Save locally first
output_file = 'cabuaur_metadata.json'

with open(output_file, 'w') as f:
    json.dump(database, f, indent=2)

print(f"✓ Saved locally: {output_file}")
print(f"  Size: {Path(output_file).stat().st_size / 1024:.1f} KB")

# Copy to Drive
import shutil
drive_output_dir = Path('/content/drive/MyDrive/RETINNA_DATA/metadata')
drive_output_dir.mkdir(parents=True, exist_ok=True)

drive_output_path = drive_output_dir / output_file
shutil.copy(output_file, drive_output_path)

print(f"\n✓ Saved to Drive: {drive_output_path}")

## Summary

In [ ]:
print("CaBuAr Metadata Extraction Complete")
print("=" * 70)

print(f"\nExtracted from HDF5:")
print(f"  ✓ 224 unique fire UUIDs")
print(f"  ✓ 424 total samples")
print(f"  ✓ Tile indices per fire")
print(f"  ✓ Train/Val/Test fold assignment")
print(f"  ✓ Comments metadata")

print(f"\nDatabase structure:")
print(f"  fires[].uuid, fire_name, location, dates, samples[]")

print(f"\nSaved to: /RETINNA_DATA/metadata/cabuaur_metadata.json")

print(f"\nNext steps:")
print(f"  1. Query Sentinel-2 API for acquisition dates")
print(f"  2. Map UUIDs to Cal Fire incidents")
print(f"  3. Extract bounding boxes from fire perimeters")
print(f"  4. Update JSON with coordinates/dates")